### retrieving doc info using RAG 
##### RAG(Retrieval-Augmented Generation)

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key = os.getenv("openai_key")
)

document =[
    "Python is a general Purpose Programming Language",
    "Learning Python is quiet easy",
    "Django is a Python web framework",
    "React supports SPA or Single Page application",
    "Learning React is fun",
    "Node is a server side platform",
    "Node.js is asynchronous",
    "PHP is a web technology",
    "Laravel is a powerful framework of PHP"
]

first embed your document into vector using llm

In [4]:
embedded_doc = []

for d in document:
    responses =client.embeddings.create(
        model="text-embedding-3-small",
        input=d
    )
    embedded_doc.append(
    {
        "text":d,
        "vector":responses.data[0].embedding
    }
)

#print(embedded_doc)
print("Embedded Document generated.")

'''but with 10,000 docs? that's 10,000 HTTP requests ..🤔
...OpenAI supports batching'''

Embedded Document generated.


"but with 10,000 docs? that's 10,000 HTTP requests ..🤔\n...OpenAI supports batching"

retrieve function

In [5]:
'''given a user's question, find the most relevant documents from your pre-embedded list.'''
from sklearn.metrics.pairwise import cosine_similarity

def retrieve(query:str):
    
    '''can't compare a raw string like "What is Python?" with document vectors right?
    need to convert the question into the same vector space'''
    query_responses = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )
    query_vector = query_responses.data[0].embedding
    
    
    '''Compute similarity against every document'''
    scores =[] #emptly list
    for it in embedded_doc:
        similarity = cosine_similarity(
            [query_vector],
            [it['vector']]
        )[0][0]
        #print(similarity)
        scores.append((similarity,it['text']))

    scores.sort(reverse=True) #put highest scores first
    #print(scores)
    top_docs =[
        text for score , text in scores[:2] # suggest best two
    ]
    return "\n".join(top_docs)



In [6]:
while True:
    user_input = input("You:")
    if user_input.lower()=='exit':
        print("Agent: Bye Bye")
        break
    context = retrieve(user_input)    
    responses = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role":"user","content":f'''
            -Context:{context}
            -Question:{user_input}
            -Use the above context to reply answer
            -Apart from the context Please reply "I dont know"
'''}
        ]
    )
    msg = responses.choices[0].message.content
    #print("Rag Context send to the AI :",context)
    print("Agent Final Reply :",msg)

    '''Basic Rag Pipe Line 
    your documents -> embedding -> vector
    question -> embedding -> vector 
    search similarity using consine similarity search 
    -> find out best relavant information 
    -> send it to llm -> final response'''

Agent Final Reply : Yes, Python is quite easy.
Agent: Bye Bye
